# ACID Transaction with a Bank Transfer

This notebook demonstrates **Atomicity, Consistency, Isolation, and Durability (ACID)** using a simple transfer of 200 from a savings account to a current account in DuckDB.

It includes table creation, test data, a successful transaction, and a failed transaction that is rolled back.

## 1. Install and import DuckDB

Run the installation cell if DuckDB is not already installed.

In [ ]:
#%pip install -q duckdb

In [1]:
import duckdb

# Create an in-memory DuckDB connection
con = duckdb.connect()


## 2. Create the accounts table and insert test data

In [2]:
con.execute("""
DROP TABLE IF EXISTS accounts;
""")

con.execute("""
CREATE TABLE accounts (
    account_id INTEGER PRIMARY KEY,
    customer_name VARCHAR,
    account_type VARCHAR,
    balance DECIMAL(10, 2)
);
""")

con.execute("""
INSERT INTO accounts (account_id, customer_name, account_type, balance)
VALUES
    (1, 'Ahmed', 'Savings', 1000.00),
    (2, 'Ahmed', 'Current', 500.00);
""")

con.sql("SELECT * FROM accounts ORDER BY account_id").show()


┌────────────┬───────────────┬──────────────┬───────────────┐
│ account_id │ customer_name │ account_type │    balance    │
│   int32    │    varchar    │   varchar    │ decimal(10,2) │
├────────────┼───────────────┼──────────────┼───────────────┤
│          1 │ Ahmed         │ Savings      │       1000.00 │
│          2 │ Ahmed         │ Current      │        500.00 │
└────────────┴───────────────┴──────────────┴───────────────┘



## 3. Successful transfer

Transfer 200 from Savings to Current. Both updates are committed together.

In [3]:
con.execute("BEGIN TRANSACTION;")

try:
    con.execute("""
    UPDATE accounts
    SET balance = balance - 200
    WHERE account_id = 1;
    """)

    con.execute("""
    UPDATE accounts
    SET balance = balance + 200
    WHERE account_id = 2;
    """)

    con.execute("COMMIT;")
    print("Transfer committed successfully.")
except Exception:
    con.execute("ROLLBACK;")
    raise

con.sql("SELECT account_type, balance FROM accounts ORDER BY account_id").show()


Transfer committed successfully.
┌──────────────┬───────────────┐
│ account_type │    balance    │
│   varchar    │ decimal(10,2) │
├──────────────┼───────────────┤
│ Savings      │        800.00 │
│ Current      │        700.00 │
└──────────────┴───────────────┘



Expected balances after the successful transfer:

| Account | Balance |
|---|---:|
| Savings | 800.00 |
| Current | 700.00 |
| Total | 1,500.00 |

## 4. Reset the test data

Restore the original balances before demonstrating rollback.

In [4]:
con.execute("""
UPDATE accounts
SET balance = CASE
    WHEN account_id = 1 THEN 1000.00
    WHEN account_id = 2 THEN 500.00
END;
""")

con.sql("SELECT account_type, balance FROM accounts ORDER BY account_id").show()


┌──────────────┬───────────────┐
│ account_type │    balance    │
│   varchar    │ decimal(10,2) │
├──────────────┼───────────────┤
│ Savings      │       1000.00 │
│ Current      │        500.00 │
└──────────────┴───────────────┘



## 5. Simulate a failure and roll back

The first update withdraws 200 from Savings. The second statement deliberately references a nonexistent column, causing an error. The exception handler rolls back the entire transaction.

In [5]:
con.execute("BEGIN TRANSACTION;")

try:
    # Step 1: Withdraw 200 from Savings
    con.execute("""
    UPDATE accounts
    SET balance = balance - 200
    WHERE account_id = 1;
    """)

    # Step 2: Deliberately cause an error
    con.execute("""
    UPDATE accounts
    SET balance = balance + 200
    WHERE account_id = 2
      AND invalid_column = 1;
    """)

    con.execute("COMMIT;")
except Exception as error:
    print(f"Simulated failure: {error}")
    con.execute("ROLLBACK;")
    print("Transaction rolled back.")

con.sql("SELECT account_type, balance FROM accounts ORDER BY account_id").show()


Simulated failure: Binder Error: Referenced column "invalid_column" not found in FROM clause!
Candidate bindings: "account_id", "balance"

LINE 5:       AND invalid_column = 1;
                  ^
Transaction rolled back.
┌──────────────┬───────────────┐
│ account_type │    balance    │
│   varchar    │ decimal(10,2) │
├──────────────┼───────────────┤
│ Savings      │       1000.00 │
│ Current      │        500.00 │
└──────────────┴───────────────┘



## 6. Verify the rollback

The balances should be restored to their original values:

| Account | Balance |
|---|---:|
| Savings | 1,000.00 |
| Current | 500.00 |
| Total | 1,500.00 |

The withdrawal was undone because the transaction was rolled back.

In [6]:
result = con.sql("""
SELECT account_type, balance
FROM accounts
ORDER BY account_id;
""").df()

display(result)


,account_type,balance
0,Savings,1000.0
1,Current,500.0


## ACID properties illustrated

- **Atomicity:** Both updates succeed, or neither is applied.
- **Consistency:** The total balance remains **1,500.00**.
- **Isolation:** Uncommitted changes are hidden from other transactions.
- **Durability:** Committed changes persist even after a system failure.